In [1]:
from dataLoader import SignLanguageDataLoader, load_data_and_labels
import numpy as np
from pathlib import Path

# Load your npz files (using the no-mask version)
dataset_path = r"D:\GP\Feature Engineering\features\splitted_data\train"
loader = load_data_and_labels(dataset_path)

# First, inspect what's inside your npz files
loader.inspect_npz_file(0)  # Check first file

# ⚠️ WARNING: This loads ALL data into memory
print("\n⚠️ Loading ALL data into memory...")
all_data, all_labels = loader.load_all()

print(f"\n✅ All data loaded!")
print(f"   Total samples: {len(all_data)}")
print(f"   Data shape of first sample: {all_data[0].shape}")
print(f"   Labels (first 5): {all_labels[:5]}")
print(f"   Unique classes: {len(set(all_labels))}")

📦 Loading metadata from cache...
✅ Loaded metadata for 9863 files

🔍 Inspecting: 0.npz
   Keys in file: ['x', 'y']
   x: shape=(206, 438), dtype=float32
   y: shape=(), dtype=int64
      Label value: 0

⚠️ Loading ALL data into memory...
⚠️ WARNING: Loading all data into memory. This may crash if dataset is large!


Loading all data: 100%|██████████| 9863/9863 [00:23<00:00, 428.77it/s]


✅ All data loaded!
   Total samples: 9863
   Data shape of first sample: (206, 438)
   Labels (first 5): [0, 0, 0, 5, 53]
   Unique classes: 441


In [2]:
# ============================================================
# APPLY AUGMENTATION TO YOUR LOADED DATA (NO MASKS)
# ============================================================

from Augemnt import MemoryEfficientAugmenter, simple_augment, augment_dataset
import numpy as np
from collections import Counter
import gc
import random

# ============================================================
# FUNCTION TO APPLY AUGMENTATION TO LIST (NO MASKS)
# ============================================================

def apply_augmentation_to_list(data, labels, 
                               augmentations_per_sample=2, 
                               augmentation_types=['shift', 'scale'], 
                               preserve_original=True,
                               verbose=True):
    """
    Apply augmentation to list of arrays (no masks)
    
    Args:
        data: List of data arrays
        labels: List of labels
        augmentations_per_sample: Number of augmentations per sample
        augmentation_types: List of augmentation types
        preserve_original: Whether to keep original samples
        verbose: Print progress
    
    Returns:
        augmented_data, augmented_labels
    """
    augmenter = MemoryEfficientAugmenter()
    
    augmented_data = []
    augmented_labels = []
    
    num_samples = len(data)
    
    if verbose:
        print(f"Processing {num_samples} samples...")
    
    for i in range(num_samples):
        sample = data[i]
        label = labels[i]
        
        # Ensure proper dtype
        if not isinstance(sample, np.ndarray):
            sample = np.array(sample, dtype=np.float32)
        else:
            sample = sample.astype(np.float32)
        
        # Add original
        if preserve_original:
            augmented_data.append(sample)
            augmented_labels.append(label)
        
        # Create augmentations
        for aug_idx in range(augmentations_per_sample):
            # Randomly select augmentation types
            if len(augmentation_types) > 2:
                selected = random.sample(augmentation_types, random.randint(1, 2))
            else:
                selected = augmentation_types
            
            # Apply augmentation (no mask)
            aug_sample = augmenter.augment_single(sample, selected)
            
            augmented_data.append(aug_sample)
            augmented_labels.append(label)
        
        # Progress
        if verbose and (i + 1) % 500 == 0:
            print(f"   Processed {i+1}/{num_samples} samples")
        
        # Clear memory periodically
        if (i + 1) % 1000 == 0:
            gc.collect()
    
    if verbose:
        print(f"\n📊 Augmentation Stats:")
        print(f"   Original samples: {num_samples}")
        print(f"   Augmented samples: {len(augmented_data)}")
        print(f"   Augmentation factor: {len(augmented_data)/num_samples:.1f}x")
    
    # Keep as lists (don't convert to 3D array to avoid inhomogeneous issues)
    return augmented_data, np.array(augmented_labels)


# ============================================================
# FUNCTION TO PAD VARIABLE-LENGTH SEQUENCES (NO MASKS)
# ============================================================

def pad_sequences_to_fixed_length(data, target_length=125, feature_dim=438):
    """
    Pad variable-length sequences to fixed length (no masks needed)
    
    Args:
        data: List of data arrays
        target_length: Desired sequence length
        feature_dim: Feature dimension
    
    Returns:
        padded_data
    """
    padded_data = []
    
    print(f"Padding {len(data)} sequences to {target_length} frames...")
    
    for i, seq in enumerate(data):
        current_len = seq.shape[0]
        
        if current_len >= target_length:
            # Truncate
            padded_seq = seq[:target_length]
        else:
            # Pad with zeros
            pad_len = target_length - current_len
            pad = np.zeros((pad_len, feature_dim))
            padded_seq = np.vstack([seq, pad])
        
        padded_data.append(padded_seq)
        
        if (i + 1) % 1000 == 0:
            print(f"   Padded {i+1}/{len(data)} sequences")
    
    # Convert to numpy array
    X = np.array(padded_data, dtype=np.float32)
    
    print(f"✅ Padding complete!")
    print(f"   X shape: {X.shape}")
    
    return X


# ============================================================
# STEP 1: CHECK YOUR DATA FORMAT FIRST
# ============================================================

print("="*60)
print("STEP 1: Checking Data Format")
print("="*60)

print(f"Data type: {type(all_data)}")
print(f"Data length: {len(all_data)}")
print(f"First sample type: {type(all_data[0])}")
print(f"First sample shape: {all_data[0].shape}")
print(f"First sample dtype: {all_data[0].dtype}")
print(f"Labels length: {len(all_labels)}")
print(f"First label: {all_labels[0]}")


# ============================================================
# STEP 2: APPLY AUGMENTATION (NO MASKS)
# ============================================================

print("\n" + "="*60)
print("STEP 2: Applying Augmentation (No Masks)")
print("="*60)

# Choose augmentation type and count
AUGMENTATIONS_PER_SAMPLE = 2
AUGMENTATION_TYPES = ['shift', 'scale']  # Use only shift and scale (safe)
PRESERVE_ORIGINAL = True

print(f"Augmentations per sample: {AUGMENTATIONS_PER_SAMPLE}")
print(f"Augmentation types: {AUGMENTATION_TYPES}")
print(f"Preserve original: {PRESERVE_ORIGINAL}")

print("\n📦 Running augmentation on list...")
X_augmented, y_augmented = apply_augmentation_to_list(
    data=all_data,
    labels=all_labels,
    augmentations_per_sample=AUGMENTATIONS_PER_SAMPLE,
    augmentation_types=AUGMENTATION_TYPES,
    preserve_original=PRESERVE_ORIGINAL,
    verbose=True
)

print(f"\n✅ Augmentation complete!")
print(f"   Original samples: {len(all_data)}")
print(f"   Augmented samples: {len(X_augmented)}")
print(f"   Augmentation factor: {len(X_augmented)/len(all_data):.1f}x")




STEP 1: Checking Data Format
Data type: <class 'list'>
Data length: 9863
First sample type: <class 'numpy.ndarray'>
First sample shape: (206, 438)
First sample dtype: float32
Labels length: 9863
First label: 0

STEP 2: Applying Augmentation (No Masks)
Augmentations per sample: 2
Augmentation types: ['shift', 'scale']
Preserve original: True

📦 Running augmentation on list...
Processing 9863 samples...
   Processed 500/9863 samples
   Processed 1000/9863 samples
   Processed 1500/9863 samples
   Processed 2000/9863 samples
   Processed 2500/9863 samples
   Processed 3000/9863 samples
   Processed 3500/9863 samples
   Processed 4000/9863 samples
   Processed 4500/9863 samples
   Processed 5000/9863 samples
   Processed 5500/9863 samples
   Processed 6000/9863 samples
   Processed 6500/9863 samples
   Processed 7000/9863 samples
   Processed 7500/9863 samples
   Processed 8000/9863 samples
   Processed 8500/9863 samples
   Processed 9000/9863 samples
   Processed 9500/9863 samples

📊 Augm

NameError: name 'augmented_class_counts' is not defined

In [6]:
from pathlib import Path
import numpy as np
from tqdm import tqdm
from collections import defaultdict
import pickle

# ============================================================
# FIXED: SAVE AUGMENTED DATA MATCHING THE LOADER FORMAT
# ============================================================

# Create output directory
output_dir = Path("trainA")
output_dir.mkdir(exist_ok=True, parents=True)

print("="*60)
print("SAVING AUGMENTED DATA WITH LABELS (LOADER COMPATIBLE)")
print("="*60)
print(f"Total samples to save: {len(X_augmented)}")
print(f"Output directory: {output_dir}/")

# ============================================================
# STEP 1: Create label encoding (convert string labels to integers)
# ============================================================

# Get unique labels from y_augmented
unique_labels = sorted(set(y_augmented))
label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

print(f"\n📊 Label encoding:")
print(f"   Unique classes: {len(unique_labels)}")
print(f"   First 10 mappings: {dict(list(label_to_idx.items())[:10])}")

# ============================================================
# STEP 2: Save each sample with correct keys 'x' and 'y'
# ============================================================

# Save each sample as a separate file with index filename (0.npz, 1.npz, etc.)
for idx, (data, label_str) in enumerate(tqdm(zip(X_augmented, y_augmented), total=len(X_augmented))):
    # Convert to numpy if it's a torch tensor
    if hasattr(data, 'cpu'):
        data = data.cpu().numpy()
    
    # Convert string label to integer using label_to_idx
    label_int = label_to_idx[label_str]
    
    # Save with keys 'x' and 'y' (matches loader expectation)
    np.savez_compressed(
        output_dir / f"{idx}.npz",
        x=data.astype(np.float32),
        y=label_int
    )

print(f"\n✅ Saved {len(X_augmented)} files to {output_dir}/")

# ============================================================
# STEP 3: Save label encoder (matches loader expectation)
# ============================================================

np.save(output_dir / "label_encoder.npy", label_to_idx)

print(f"✅ Saved label encoder to {output_dir}/label_encoder.npy")

# ============================================================
# VERIFICATION: Check if saved files match loader format
# ============================================================

print("\n" + "="*60)
print("VERIFICATION: Checking saved files")
print("="*60)

# Check first file
first_file = output_dir / "0.npz"
if first_file.exists():
    test_load = np.load(first_file)
    print(f"\n✅ First file: {first_file}")
    print(f"   Keys: {list(test_load.keys())}")
    print(f"   'x' shape: {test_load['x'].shape}")
    print(f"   'y' value: {test_load['y']} (type: {type(test_load['y'])})")
    
    # Verify we can decode the label
    loaded_label_int = int(test_load['y'])
    loaded_label_str = idx_to_label[loaded_label_int]
    print(f"   Decoded label: {loaded_label_str}")
    
    # Check if keys match expected ('x' and 'y')
    expected_keys = {'x', 'y'}
    actual_keys = set(test_load.keys())
    if expected_keys.issubset(actual_keys):
        print(f"   ✅ Keys match! ('x' and 'y' present)")
    else:
        print(f"   ❌ Keys don't match! Expected {expected_keys}, got {actual_keys}")
    
    test_load.close()
else:
    print(f"❌ First file not found: {first_file}")

# ============================================================
# STEP 4: Check label encoder
# ============================================================

print(f"\n📊 Label encoder contents:")
print(f"   Total classes: {len(label_to_idx)}")
print(f"   Sample mappings:")
for label, idx in list(label_to_idx.items())[:10]:
    print(f"      '{label}' -> {idx}")

# ============================================================
# STEP 5: Summary
# ============================================================

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Output directory: {output_dir}/")
print(f"Files saved: {len(X_augmented)}")
print(f"File naming: 0.npz, 1.npz, 2.npz, ...")
print(f"Each file contains:")
print(f"   - 'x': data array (shape: {X_augmented[0].shape if X_augmented else 'N/A'})")
print(f"   - 'y': integer label (0 to {len(unique_labels)-1})")
print(f"Label encoder saved: label_encoder.npy")
print("\n✅ Ready to be loaded by the split script!")

SAVING AUGMENTED DATA WITH LABELS (LOADER COMPATIBLE)
Total samples to save: 29589
Output directory: trainA/

📊 Label encoding:
   Unique classes: 441
   First 10 mappings: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9}


100%|██████████| 29589/29589 [05:01<00:00, 98.11it/s] 


✅ Saved 29589 files to trainA/
✅ Saved label encoder to trainA/label_encoder.npy

VERIFICATION: Checking saved files

✅ First file: trainA\0.npz
   Keys: ['x', 'y']
   'x' shape: (206, 438)
   'y' value: 0 (type: <class 'numpy.ndarray'>)
   Decoded label: 0
   ✅ Keys match! ('x' and 'y' present)

📊 Label encoder contents:
   Total classes: 441
   Sample mappings:
      '0' -> 0
      '1' -> 1
      '2' -> 2
      '3' -> 3
      '4' -> 4
      '5' -> 5
      '6' -> 6
      '7' -> 7
      '8' -> 8
      '9' -> 9

SUMMARY
Output directory: trainA/
Files saved: 29589
File naming: 0.npz, 1.npz, 2.npz, ...
Each file contains:
   - 'x': data array (shape: (206, 438))
   - 'y': integer label (0 to 440)
Label encoder saved: label_encoder.npy

✅ Ready to be loaded by the split script!
